In [97]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix


df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

train_df, remaining_df = train_test_split(
    df, test_size=0.4, stratify=df["Churn"], random_state=42
)

valid_df, test_df = train_test_split(remaining_df,stratify=remaining_df["Churn"],test_size=0.5,random_state=42)
train_df["TotalCharges"] = pd.to_numeric(train_df["TotalCharges"].str.strip().replace("", np.nan), errors="raise")
valid_df["TotalCharges"] = pd.to_numeric(valid_df["TotalCharges"].str.strip().replace("", np.nan), errors="raise")
test_df["TotalCharges"] = pd.to_numeric(test_df["TotalCharges"].str.strip().replace("", np.nan), errors="raise")


first_features =["tenure","MonthlyCharges"]

X_train = train_df[first_features]
y_train = train_df["Churn"].map({"No": 0, "Yes": 1})

first_model = LogisticRegression(max_iter=1000)
first_model.fit(X_train,y_train)


X_valid = valid_df[first_features]
y_valid = valid_df["Churn"].map({"No": 0, "Yes": 1})

pred = first_model.predict(X_valid)

print("Logistic Regression\n")
print("Accuracy score : ", accuracy_score(y_valid,pred))
print("Confusion matrix : ", confusion_matrix(y_valid,pred,labels=[0,1]))






Logistic Regression

Accuracy score :  0.7828246983676366
Confusion matrix :  [[950  85]
 [221 153]]


In [98]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train,y_train)
pred_baseline = baseline.predict(X_valid)

print("DummyClassifier\n")
print("Accuracy score : ", accuracy_score(y_valid,pred_baseline))
print("Confusion matrix : ", confusion_matrix(y_valid,pred_baseline,labels=[0,1]))

DummyClassifier

Accuracy score :  0.7345635202271115
Confusion matrix :  [[1035    0]
 [ 374    0]]


In [99]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np


numeric_features = ["tenure","MonthlyCharges","TotalCharges"]
categorical_features = ["Contract","InternetService","PaymentMethod"]

numeric_transformer = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler()) 
     ])

categorical_transformer = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore")) 
     ])

preprocessor = ColumnTransformer([
    ("num",numeric_transformer,numeric_features),
    ("cat",categorical_transformer,categorical_features)
    ])

pipe = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",LogisticRegression(max_iter=1000))
    ])


feature_cols = numeric_features + categorical_features

X_train = train_df[feature_cols]
y_train = train_df["Churn"].map({"No": 0, "Yes": 1})
X_valid = valid_df[feature_cols]

pipe.fit(X_train,y_train)
pred_pipe = pipe.predict(X_valid)
print(pred_pipe)


[0 0 0 ... 0 0 0]


In [100]:
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

numeric_features = ["tenure","MonthlyCharges","TotalCharges"]
categorical_features = ["Contract","InternetService","PaymentMethod"]

def build_pipeline(classifier) :
    
    numeric_transformer = Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()) 
        ])

    categorical_transformer = Pipeline([
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore")) 
        ])

    preprocessor = ColumnTransformer([
        ("num",numeric_transformer,numeric_features),
        ("cat",categorical_transformer,categorical_features)
        ])

    pipe = Pipeline([
        ("preprocessor",preprocessor),
        ("classifier",classifier)
        ])
    return pipe 

def evaluate_model(model, X , y ) :
    threshold = 0.5
    indice = list(model.classes_).index(1)

    pred_proba = model.predict_proba(X)

    scores = pred_proba[:,indice]
    pred_eval =[ 1 if score >= threshold else 0 for score in scores]


    accuracy = accuracy_score(y,pred_eval)
    f1 = f1_score(y,pred_eval,zero_division=0)
    recall = recall_score(y,pred_eval,zero_division=0)
    matrix = confusion_matrix(y,pred_eval,labels=[0,1])
    avg_precision = average_precision_score(y,scores)
    precision = precision_score(y,pred_eval,zero_division=0)

    return {
        "accuracy" : float(accuracy),
        "recall" : float(recall),
        "f1_score" : float(f1),
        "avg_precision" : float(avg_precision),
        "matrix_confusion" : matrix.tolist(),
        "precision" : float(precision),
        "threshold" : threshold
    }
    




    

In [ ]:
#Test 
import json
from sklearn.dummy import DummyClassifier

baseline_pipe = build_pipeline(DummyClassifier(strategy="most_frequent"))
baseline_pipe.fit(X_train, y_train)
print(json.dumps(evaluate_model(baseline_pipe, X_valid, y_valid)))

logreg_pipe = build_pipeline(LogisticRegression(max_iter=1000))
logreg_pipe.fit(X_train, y_train)
print(json.dumps(evaluate_model(logreg_pipe, X_valid, y_valid)))

{"accuracy": 0.7345635202271115, "recall": 0.0, "f1_score": 0.0, "avg_precision": 0.2654364797728886, "matrix_confusion": [[1035, 0], [374, 0]], "precision": 0.0, "threshold": 0.5}
{"accuracy": 0.8041163946061036, "recall": 0.5294117647058824, "f1_score": 0.5892857142857143, "avg_precision": 0.6474664703085159, "matrix_confusion": [[935, 100], [176, 198]], "precision": 0.6644295302013423, "threshold": 0.5}


In [121]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import time
from pathlib import Path


models = {
    "LogisticRegression" : LogisticRegression(max_iter=1000),
    "RandomForestClassifier" : RandomForestClassifier(n_estimators=100,random_state=42,max_depth=8),
    "DecisionTreeClassifier" : DecisionTreeClassifier(random_state=42,max_depth=5),
    "DummyClassifier" :DummyClassifier(strategy="most_frequent")
    }

result = []
output = Path("../reports/train.json")

for name,model in models.items() :
    candidate = build_pipeline(model)
    start = time.perf_counter()
    candidate.fit(X_train,y_train)
    end = time.perf_counter()
    diff = end -start

    metrics = evaluate_model(candidate,X_valid,y_valid)

    result.append(
        {"model" : name, **metrics,
         "params" : model.get_params() , 
         "train_time" : round(diff,4)})

output.parent.mkdir(parents=True,exist_ok=True)
output.write_text(json.dumps(result,indent=2))
print(result)

[{'model': 'LogisticRegression', 'accuracy': 0.8041163946061036, 'recall': 0.5294117647058824, 'f1_score': 0.5892857142857143, 'avg_precision': 0.6474664703085159, 'matrix_confusion': [[935, 100], [176, 198]], 'precision': 0.6644295302013423, 'threshold': 0.5, 'params': {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}, 'train_time': 0.0394}, {'model': 'RandomForestClassifier', 'accuracy': 0.794889992902768, 'recall': 0.5080213903743316, 'f1_score': 0.5680119581464873, 'avg_precision': 0.6699512606026616, 'matrix_confusion': [[930, 105], [184, 190]], 'precision': 0.6440677966101694, 'threshold': 0.5, 'params': {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min